# Pseudobulk model building — GenomicSuperSignature (GSS)

**Environment:** `clamp-analyses`

Builds a GenomicSuperSignature RAVmodel from PCA loadings for every pseudobulk dataset. Loadings are hierarchically clustered into Replicable Axes of Variation (RAVs). The RAV × sample score matrix is saved as B. Mirrors `nbs/01_model_building/02_gtex/09_GenomicSuperSignature.ipynb`. Outputs written to `output/01_model_building/05_pseudobulk/<dataset>/GSS/`.

## Libraries

In [ ]:
library(data.table)
library(here)
library(CLAMP)
library(matrixStats)
library(factoextra)
library(cluster)
library(GenomicSuperSignature)

set.seed(123)

## Configuration

In [ ]:
DATASET   = "PBMC_Perez2022"
D_CLUSTER = 4L
OUT_ROOT  = "output/01_model_building/05_pseudobulk"
DATA_DIR  = "data/pseudobulk"

## Build GSS model for each dataset

In [ ]:
message("========== ", DATASET, " ==========")
out_dir <- file.path(here(), OUT_ROOT, DATASET)

# Load preprocessed data
norm_dt    <- fread(file.path(here(), OUT_ROOT, DATASET, "norm.csv"))
norm_genes <- norm_dt[[1]]
norm       <- as.matrix(norm_dt[, -1, with = FALSE])
storage.mode(norm) <- "numeric"
rownames(norm) <- norm_genes
samples <- colnames(norm)
cat(DATASET, "norm:", nrow(norm), "genes x", ncol(norm), "samples\n")

# Load k
n_pcs <- as.integer(read.csv(file.path(here(), OUT_ROOT, DATASET, "k.csv"))$k[1])
message("  n_pcs = ", n_pcs)

# PCA
expr_mat <- as.matrix(norm)
storage.mode(expr_mat) <- "double"
pca_res <- prcomp(t(expr_mat))

study <- DATASET
trainingData_PCA <- list()
trainingData_PCA[[study]] <- list()
trainingData_PCA[[study]]$rotation <- pca_res$rotation[, 1:n_pcs]
colnames(trainingData_PCA[[study]]$rotation) <- paste0(study, ".PC", 1:n_pcs)

eigs <- pca_res$sdev^2
pca_summary <- rbind(
  SD         = sqrt(eigs),
  Variance   = eigs / sum(eigs),
  Cumulative = cumsum(eigs) / sum(eigs)
)
trainingData_PCA[[study]]$variance <- pca_summary[, 1:n_pcs]
colnames(trainingData_PCA[[study]]$variance) <- paste0(study, ".PC", 1:n_pcs)

# Hierarchical clustering of PC loadings -> RAVs
allZ <- trainingData_PCA[[study]]$rotation
storage.mode(allZ) <- "double"
all  <- t(allZ)

k_clust <- max(round(nrow(all) / D_CLUSTER, 0), 2)
cat("  n_clusters (RAVs):", k_clust, "\n")

res.dist <- as.dist(1 - cor(t(all), method = "spearman"))
hc       <- hclust(res.dist, method = "ward.D")
cl_vec   <- cutree(hc, k = k_clust)

trainingData_PCclusters <- buildAvgLoading(allZ, k_clust,
                                           cluster = cl_vec)

cl <- trainingData_PCclusters$cluster
silh_res <- tryCatch(
  cluster::silhouette(cl, res.dist),
  error = function(e) NULL
)
trainingData_PCclusters$sw <- if (!is.null(silh_res)) {
  tryCatch(
    summary(silh_res)$clus.avg.widths,
    error = function(e) rep(NA_real_, k_clust)
  )
} else {
  rep(NA_real_, k_clust)
}

# Build RAVmodel
trainingData_df <- DataFrame(
  PCAsummary = I(list(trainingData_PCA[[study]]$variance))
)
rownames(trainingData_df) <- study

RAVmodel <- PCAGenomicSignatures(
  assays       = list(RAVindex = as.matrix(trainingData_PCclusters$avgLoading)),
  trainingData = trainingData_df
)
metadata(RAVmodel) <- trainingData_PCclusters[c("cluster", "size", "k", "n")]
names(metadata(RAVmodel)$size) <- paste0("RAV", seq_len(ncol(RAVmodel)))
geneSets(RAVmodel)        <- "Custom"
studies(RAVmodel)         <- trainingData_PCclusters$studies
silhouetteWidth(RAVmodel) <- trainingData_PCclusters$sw
updateNote(RAVmodel)      <- paste0("Single-matrix pseudobulk model; dataset=", DATASET)
metadata(RAVmodel)$version <- "0.1.0-pseudobulk"

# Compute B (RAVs x samples) via projection
RAVindex <- assays(RAVmodel)[["RAVindex"]] |> as.matrix()
storage.mode(RAVindex) <- "double"
common <- sort(intersect(norm_genes, rownames(RAVindex)))
expr_c     <- expr_mat[common, , drop = FALSE]
RAVindex_c <- RAVindex[common, , drop = FALSE]
B_RAVxSample <- crossprod(RAVindex_c, expr_c)
colnames(B_RAVxSample) <- samples
rownames(B_RAVxSample) <- paste0("RAV", seq_len(nrow(B_RAVxSample)))

Z_mat <- RAVindex_c
colnames(Z_mat) <- paste0("RAV", seq_len(ncol(Z_mat)))

model_dir <- file.path(out_dir, "GSS")
dir.create(model_dir, showWarnings = FALSE, recursive = TRUE)
write.csv(as.data.frame(B_RAVxSample), file.path(model_dir, "B.csv"))
write.csv(as.data.frame(Z_mat),        file.path(model_dir, "Z.csv"))
saveRDS(RAVmodel, file.path(model_dir, "RAVmodel.rds"))
message("  GSS saved -> ", model_dir)